# RF-DETR Weight-Sharing NAS – Google Colab Runner

Notebook điều phối để kéo code từ GitHub về Google Colab (GPU **L4**), cài đặt package `rfdetr_reproduction`, và chạy toàn bộ 3 giai đoạn reproduction + **thí nghiệm cải tiến A/B (PAB-NAS / Sandwich)** với **checkpoint & kết quả đồng bộ trực tiếp với cấu trúc Google Drive hiện có**:

```text
/content/drive/MyDrive/RFDETR_Checkpoints/
├── nas_supernet_cocoinit_full.pth          (checkpoint Baseline Uniform Epoch 20 có sẵn)
├── nas_train_history.csv
├── project_checkpoints/nas/
│   ├── nas_supernet_latest.pth             (symlink tới bản Uniform)
│   ├── nas_supernet_ema_only.pth
│   ├── nas_supernet_pab_nas_latest.pth     (checkpoint cải tiến tách riêng, không ghi đè)
│   └── nas_supernet_pab_nas_ema_only.pth
└── results_cocoinit/                       (thư mục lưu kết quả & biểu đồ)
    ├── nas/
    ├── diagnostics/
    ├── grid_search/
    ├── deployment/
    └── baselines/
```


## 1. Mount Google Drive & Pull/Clone Code từ GitHub


In [ ]:
import os
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# === CẤU HÌNH REPO GITHUB CỦA BẠN ===
GITHUB_REPO_URL = "https://github.com/<USERNAME>/<TEN_REPO>.git"  # <-- Điền link repo của bạn
BRANCH = "main"
USE_PRIVATE_TOKEN = False  # Đổi thành True nếu repo private (lưu PAT vào Colab Secrets tên GITHUB_TOKEN)

REPO_DIR = Path("/content/RF-DETR-Reproduction")

repo_url = GITHUB_REPO_URL
if USE_PRIVATE_TOKEN:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    repo_url = GITHUB_REPO_URL.replace("https://", f"https://{token}@")

if (REPO_DIR / ".git").exists():
    print(f"Repo đã tồn tại tại {REPO_DIR} -> chạy git pull ({BRANCH})...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    print(f"Cloning {GITHUB_REPO_URL} (branch={BRANCH}) -> {REPO_DIR} ...")
    subprocess.run(["git", "clone", "-b", BRANCH, repo_url, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## 2. Cài đặt thư viện, Package `rfdetr_reproduction` & Chạy Unit Tests


In [ ]:
!pip install -q "rfdetr[train,loggers] @ git+https://github.com/roboflow/rf-detr.git@382607f3d7ecbf513b991a8ee9f1a3abe3a340f0"
!pip install -q pycocotools onnx onnxruntime-gpu ultralytics pytest tqdm
!pip install -q -e .

# Chạy unit test kiểm tra validator, search space (111 / 3948 / 3780), dataset & PAB-NAS sampler
!pytest -v


## 3. Ánh xạ cấu trúc Google Drive hiện có (`RFDETR_Checkpoints`) vào Project

Cell này tự động:
1. Trỏ `RFDETR_DATA_DIR` về SSD local của Colab (`/content/data`) để đọc ảnh nhanh nhất.
2. Trỏ `RFDETR_CKPT_DIR` và `RFDETR_OUTPUT_DIR` thẳng vào `/content/drive/MyDrive/RFDETR_Checkpoints/`.
3. **Liên kết tự động checkpoint sẵn có** `nas_supernet_cocoinit_full.pth` → `nas/nas_supernet_latest.pth` và tách sẵn `nas/nas_supernet_ema_only.pth` (nhẹ hơn cho Grid Search & Export) để mọi script nhận ngay trọng số Epoch 20 mà không phải train lại từ đầu.
4. **Liên kết tự động các file kết quả cũ** (`nas_validation_results_cocoinit.csv`, `pareto_frontier_cocoinit.csv`, `nas_train_history.csv`) vào thư mục đích để có thể resume mượt mà.


In [ ]:
import os
import shutil
from pathlib import Path
import torch

DRIVE_ROOT = Path("/content/drive/MyDrive/RFDETR_Checkpoints")
DRIVE_RESULTS = DRIVE_ROOT / "results_cocoinit"
DRIVE_FULL_CKPT = DRIVE_ROOT / "nas_supernet_cocoinit_full.pth"

# Thư mục runtime chuẩn cho các script trong project
DATA_DIR = Path("/content/data")
CKPT_DIR = DRIVE_ROOT / "project_checkpoints"
OUTPUT_DIR = DRIVE_RESULTS

os.environ["RFDETR_DATA_DIR"] = str(DATA_DIR)
os.environ["RFDETR_CKPT_DIR"] = str(CKPT_DIR)
os.environ["RFDETR_OUTPUT_DIR"] = str(OUTPUT_DIR)

for d in [
    DATA_DIR,
    CKPT_DIR / "nas",
    CKPT_DIR / "pretrained",
    CKPT_DIR / "diagnostics",
    OUTPUT_DIR / "nas",
    OUTPUT_DIR / "diagnostics",
    OUTPUT_DIR / "grid_search",
    OUTPUT_DIR / "deployment",
    OUTPUT_DIR / "baselines",
]:
    d.mkdir(parents=True, exist_ok=True)


def _link_or_copy(src: Path, dst: Path):
    if not src.exists() or dst.exists():
        return
    try:
        os.symlink(src, dst)
        print(f"[Symlink] {dst.name} -> {src}")
    except OSError:
        shutil.copy2(src, dst)
        print(f"[Copy]    {src} -> {dst}")


# 1. Liên kết checkpoint Epoch 20 sẵn có vào chuẩn project
nas_latest = CKPT_DIR / "nas" / "nas_supernet_latest.pth"
nas_ema = CKPT_DIR / "nas" / "nas_supernet_ema_only.pth"

if DRIVE_FULL_CKPT.exists():
    _link_or_copy(DRIVE_FULL_CKPT, nas_latest)
    if not nas_ema.exists():
        print("Đang tách bản nhẹ nas_supernet_ema_only.pth từ nas_supernet_cocoinit_full.pth ...")
        ckpt = torch.load(DRIVE_FULL_CKPT, map_location="cpu", weights_only=False)
        torch.save(
            {
                "epoch": ckpt.get("epoch", 20),
                "ema_model": ckpt.get("ema_model", ckpt.get("model")),
                "ema_updates": ckpt.get("ema_updates", 0),
                "provenance": ckpt.get("provenance"),
            },
            nas_ema,
        )
        del ckpt
        print(f"[Created] {nas_ema} ({nas_ema.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"[INFO] Không tìm thấy {DRIVE_FULL_CKPT} -> sẽ train mới khi chạy Stage 1.")

# 2. Liên kết các file kết quả đã có trên Drive để resume Grid Search / Deployment
_link_or_copy(DRIVE_RESULTS / "nas_validation_results_cocoinit.csv", OUTPUT_DIR / "grid_search" / "nas_validation_results.csv")
_link_or_copy(DRIVE_RESULTS / "pareto_frontier_cocoinit.csv", OUTPUT_DIR / "grid_search" / "pareto_frontier.csv")
_link_or_copy(DRIVE_RESULTS / "rfdetr_nas_supernet" / "nas_train_history.csv", OUTPUT_DIR / "nas" / "nas_train_history.csv")

print("\n=== TRẠNG THÁI ĐƯỜNG DẪN ===")
print("RFDETR_DATA_DIR  =", os.environ["RFDETR_DATA_DIR"])
print("RFDETR_CKPT_DIR  =", os.environ["RFDETR_CKPT_DIR"])
print("RFDETR_OUTPUT_DIR=", os.environ["RFDETR_OUTPUT_DIR"])
print("Latest ckpt      =", nas_latest.exists(), f"({nas_latest})")
print("EMA-only ckpt    =", nas_ema.exists(), f"({nas_ema})")


## 4. Giai đoạn 1 – Kiểm tra môi trường, Chuẩn bị VOC2007 & Huấn luyện NAS (Baseline Uniform)


In [ ]:
# 4.1. Ghi provenance (GPU L4, CUDA, PyTorch, Git SHA) -> results_cocoinit/provenance.json
!python scripts/setup_environment.py --require-cuda

# 4.2. Tải VOC2007 & chuyển sang COCO JSON (lọc difficult=1, kiểm tra zero-overlap)
!python scripts/prepare_dataset.py


In [ ]:
# 4.3. Huấn luyện Weight-Sharing NAS Supernet (Baseline Uniform)
# Mặc định đặt False vì bạn đã có sẵn nas_supernet_cocoinit_full.pth (Epoch 20) trên Drive.
RUN_TRAIN_NAS = False
NAS_EPOCHS = 20

if RUN_TRAIN_NAS:
    !python scripts/train_nas.py --sampling-strategy uniform --epochs {NAS_EPOCHS}
    if not DRIVE_FULL_CKPT.is_symlink() and (CKPT_DIR / "nas" / "nas_supernet_latest.pth").exists():
        shutil.copy2(CKPT_DIR / "nas" / "nas_supernet_latest.pth", DRIVE_FULL_CKPT)
    hist_src = OUTPUT_DIR / "nas" / "nas_train_history.csv"
    if hist_src.exists():
        shutil.copy2(hist_src, DRIVE_ROOT / "nas_train_history.csv")
        (DRIVE_RESULTS / "rfdetr_nas_supernet").mkdir(parents=True, exist_ok=True)
        shutil.copy2(hist_src, DRIVE_RESULTS / "rfdetr_nas_supernet" / "nas_train_history.csv")
else:
    print("Bỏ qua train_nas.py (đang dùng sẵn checkpoint Baseline Uniform Epoch 20 trên Drive).")


## 5. Giai đoạn 2 – Thí nghiệm Chẩn đoán (Gradient Conflict, Hypo 1, Hypo 2) & Grid Search


In [ ]:
# 5a. Gradient Conflict Diagnosis (Mục 5b)
from IPython.display import Image, display
import pandas as pd

RUN_GRADIENT_CONFLICT = False
GC_TAG = "ep20"

if RUN_GRADIENT_CONFLICT:
    !python scripts/run_gradient_conflict.py --checkpoint "{DRIVE_FULL_CKPT}" --tag {GC_TAG}
    img_gc = OUTPUT_DIR / "diagnostics" / f"gradient_conflict_heatmap_{GC_TAG}.png"
    csv_gc = OUTPUT_DIR / "diagnostics" / f"gradient_conflict_summary_{GC_TAG}.csv"
    if csv_gc.exists():
        display(pd.read_csv(csv_gc).round(4))
    if img_gc.exists():
        display(Image(filename=str(img_gc)))


In [ ]:
# 5b. Hypothesis 1 – Tốc độ hội tụ riêng của từng Architecture (Mục 5d)
# Tự động lưu & resume trạng thái qua project_checkpoints/diagnostics/hypo1_state.pth trên Drive
RUN_HYPO1 = False
HYPO1_EPOCHS = 20
HYPO1_EVAL_EVERY = 2

if RUN_HYPO1:
    !python scripts/run_hypothesis1.py --epochs {HYPO1_EPOCHS} --eval-every {HYPO1_EVAL_EVERY}
    csv_h1 = OUTPUT_DIR / "diagnostics" / "hypo1_t90_summary.csv"
    img_h1 = OUTPUT_DIR / "diagnostics" / "hypo1_convergence.png"
    if csv_h1.exists():
        display(pd.read_csv(csv_h1).round(4))
    if img_h1.exists():
        display(Image(filename=str(img_h1)))


In [ ]:
# 5c. Hypothesis 2 – Weight Drift / Interference trên mạng nhỏ khi tăng p(A) (Mục 5e)
# Nạp trực tiếp từ nas_supernet_cocoinit_full.pth (không ghi đè checkpoint gốc)
RUN_HYPO2 = False
HYPO2_EXTRA_EPOCHS = 5

if RUN_HYPO2:
    !python scripts/run_hypothesis2.py --ckpt "{DRIVE_FULL_CKPT}" --extra-epochs {HYPO2_EXTRA_EPOCHS}
    csv_h2 = OUTPUT_DIR / "diagnostics" / "hypo2_summary.csv"
    img_h2 = OUTPUT_DIR / "diagnostics" / "hypo2_interference.png"
    if csv_h2.exists():
        display(pd.read_csv(csv_h2).round(4))
    if img_h2.exists():
        display(Image(filename=str(img_h2)))


In [ ]:
# 5d. Post-Training Grid Search & Pareto Frontier (Baseline Uniform – Mục 6)
RUN_GRID_SEARCH = False
GRID_MODE = "representative"  # "representative" | "full"

if RUN_GRID_SEARCH:
    !python scripts/run_grid_search.py --mode {GRID_MODE} --checkpoint "{CKPT_DIR / 'nas' / 'nas_supernet_ema_only.pth'}"
    pareto_csv = OUTPUT_DIR / "grid_search" / "pareto_frontier.csv"
    pareto_png = OUTPUT_DIR / "grid_search" / "nas_pareto_frontier.png"
    if pareto_csv.exists():
        display(pd.read_csv(pareto_csv))
    if pareto_png.exists():
        display(Image(filename=str(pareto_png)))
else:
    print("Bỏ qua Grid Search Baseline (đã có sẵn pareto_frontier_cocoinit.csv trên Drive).")


## 6. Giai đoạn 3 – Deployment (ONNX/TensorRT/Throttling), YOLO Baselines & Segmentation


In [ ]:
RUN_DEPLOYMENT = False
RUN_BASELINES = False
RUN_SEGMENTATION = False

EMA_CKPT = CKPT_DIR / "nas" / "nas_supernet_ema_only.pth"
PARETO_CSV = OUTPUT_DIR / "grid_search" / "pareto_frontier.csv"

if RUN_DEPLOYMENT:
    !python scripts/run_deployment.py --ckpt "{EMA_CKPT}" --pareto-csv "{PARETO_CSV}"
    thr_png = OUTPUT_DIR / "deployment" / "throttling_comparison.png"
    if thr_png.exists():
        display(Image(filename=str(thr_png)))

if RUN_BASELINES:
    !python scripts/run_baselines.py --ckpt "{EMA_CKPT}" --pareto-csv "{PARETO_CSV}" --epochs 20
    comp_csv = OUTPUT_DIR / "baselines" / "fair_baseline_comparison_test.csv"
    if comp_csv.exists():
        display(pd.read_csv(comp_csv))

if RUN_SEGMENTATION:
    !python scripts/run_segmentation.py --enable


## 7. Cải tiến Phương pháp (PAB-NAS / Sandwich Micro-Batching) & Đối chứng A/B

Chạy trên **cùng điều kiện** với Baseline Uniform (`SEED=42`, COCO-init, 20 epochs trên VOC2007) và lưu tách biệt hoàn toàn theo `PAB_EXP_TAG` (không ghi đè checkpoint hay CSV của Baseline):
- `PAB_STRATEGY = "pruned_anchor"` → cắt tỉa `p=32, w=4` + 2 giai đoạn với ngân sách mỏ neo ~32% cho mạng nhỏ.
- `PAB_STRATEGY = "sandwich"` → thêm Sandwich Micro-Batching trên 4 micro-steps (`1 lớn + 1 nhỏ + 2 lõi`).


In [ ]:
# 7.1. Huấn luyện siêu mạng cải tiến (tự động resume từ Drive nếu Colab ngắt giữa chừng)
from IPython.display import Image, display
import pandas as pd

PAB_STRATEGY = "pruned_anchor"   # Đổi thành "sandwich" nếu muốn chạy bản Sandwich Micro-Batching
PAB_EXP_TAG = "pab_nas"          # Gợi ý: "pab_nas" cho pruned_anchor, "sandwich_nas" cho sandwich
PAB_EPOCHS = 20

RUN_PAB_TRAIN = True
RUN_PAB_GC = True
RUN_PAB_GRID_SEARCH = True

PAB_LATEST_CKPT = CKPT_DIR / "nas" / f"nas_supernet_{PAB_EXP_TAG}_latest.pth"
PAB_EMA_CKPT = CKPT_DIR / "nas" / f"nas_supernet_{PAB_EXP_TAG}_ema_only.pth"

if RUN_PAB_TRAIN:
    !python scripts/train_nas.py --sampling-strategy {PAB_STRATEGY} --exp-tag {PAB_EXP_TAG} --epochs {PAB_EPOCHS}
    pab_hist_csv = OUTPUT_DIR / "nas" / f"nas_train_history_{PAB_EXP_TAG}.csv"
    if pab_hist_csv.exists():
        display(pd.read_csv(pab_hist_csv).tail(5))


In [ ]:
# 7.2. Chẩn đoán Gradient Conflict & Quét lại 37 subnets Pareto cho bản cải tiến
if RUN_PAB_GC:
    assert PAB_LATEST_CKPT.exists(), f"Chưa tìm thấy checkpoint {PAB_LATEST_CKPT}"
    !python scripts/run_gradient_conflict.py --checkpoint "{PAB_LATEST_CKPT}" --tag {PAB_EXP_TAG}_ep{PAB_EPOCHS}
    gc_csv = OUTPUT_DIR / "diagnostics" / f"gradient_conflict_summary_{PAB_EXP_TAG}_ep{PAB_EPOCHS}.csv"
    gc_png = OUTPUT_DIR / "diagnostics" / f"gradient_conflict_heatmap_{PAB_EXP_TAG}_ep{PAB_EPOCHS}.png"
    if gc_csv.exists():
        display(pd.read_csv(gc_csv).round(4))
    if gc_png.exists():
        display(Image(filename=str(gc_png)))

if RUN_PAB_GRID_SEARCH:
    assert PAB_EMA_CKPT.exists(), f"Chưa tìm thấy EMA checkpoint {PAB_EMA_CKPT}"
    !python scripts/run_grid_search.py --mode representative --checkpoint "{PAB_EMA_CKPT}" --exp-tag {PAB_EXP_TAG}
    pab_pareto_csv = OUTPUT_DIR / "grid_search" / f"pareto_frontier_{PAB_EXP_TAG}.csv"
    pab_pareto_png = OUTPUT_DIR / "grid_search" / f"nas_pareto_frontier_{PAB_EXP_TAG}.png"
    if pab_pareto_csv.exists():
        display(pd.read_csv(pab_pareto_csv))
    if pab_pareto_png.exists():
        display(Image(filename=str(pab_pareto_png)))


In [ ]:
# 7.3. Bảng tổng hợp & Biểu đồ đối chứng A/B: Baseline (Uniform) vs Cải tiến (PAB-NAS / Sandwich)
import matplotlib.pyplot as plt
import pandas as pd

base_val_csv = OUTPUT_DIR / "grid_search" / "nas_validation_results.csv"
base_par_csv = OUTPUT_DIR / "grid_search" / "pareto_frontier.csv"
pab_val_csv = OUTPUT_DIR / "grid_search" / f"nas_validation_results_{PAB_EXP_TAG}.csv"
pab_par_csv = OUTPUT_DIR / "grid_search" / f"pareto_frontier_{PAB_EXP_TAG}.csv"

base_hist_csv = OUTPUT_DIR / "nas" / "nas_train_history.csv"
pab_hist_csv = OUTPUT_DIR / "nas" / f"nas_train_history_{PAB_EXP_TAG}.csv"

if base_val_csv.exists() and pab_val_csv.exists():
    b_val = pd.read_csv(base_val_csv)
    p_val = pd.read_csv(pab_val_csv)
    b_par = pd.read_csv(base_par_csv)
    p_par = pd.read_csv(pab_par_csv)

    # So sánh các mốc kiến trúc trọng tâm: Tiny (320), Nano (384), Peak Pareto (640)
    key_specs = [
        ("Tiny (320/p16/w2/d6/q300)",  (320, 16, 2, 6, 300)),
        ("Nano (384/p16/w2/d6/q300)",  (384, 16, 2, 6, 300)),
        ("Mid  (512/p16/w2/d6/q300)",  (512, 16, 2, 6, 300)),
        ("Peak (640/p16/w2/d6/q300)",  (640, 16, 2, 6, 300)),
    ]

    def _lookup(df, spec):
        r, ps, nw, dl, nq = spec
        sub = df[
            (df["resolution"] == r) & (df["patch_size"] == ps) & (df["num_windows"] == nw)
            & (df["decoder_layers"] == dl) & (df["num_queries"] == nq) & (df["valid"] == True)
        ]
        return sub.iloc[0] if len(sub) else None

    comp_rows = []
    for name, spec in key_specs:
        rb = _lookup(b_val, spec)
        rp = _lookup(p_val, spec)
        if rb is not None and rp is not None:
            comp_rows.append({
                "subnet": name,
                "uniform_mAP(%)": round(rb["map"] * 100, 2),
                f"{PAB_EXP_TAG}_mAP(%)": round(rp["map"] * 100, 2),
                "delta_mAP(%)": round((rp["map"] - rb["map"]) * 100, 2),
                "uniform_mAP50(%)": round(rb["map50"] * 100, 2),
                f"{PAB_EXP_TAG}_mAP50(%)": round(rp["map50"] * 100, 2),
            })

    ab_df = pd.DataFrame(comp_rows)
    ab_csv = OUTPUT_DIR / "grid_search" / f"ab_comparison_uniform_vs_{PAB_EXP_TAG}.csv"
    ab_df.to_csv(ab_csv, index=False)
    print("=== SO SÁNH mAP: BASELINE UNIFORM VS CẢI TIẾN ===")
    display(ab_df)

    if base_hist_csv.exists() and pab_hist_csv.exists():
        bh = pd.read_csv(base_hist_csv)
        ph = pd.read_csv(pab_hist_csv)
        print(
            f"Thời gian train trung bình/epoch: Uniform = {bh['elapsed_sec'].mean()/60:.2f} phút | "
            f"{PAB_EXP_TAG} = {ph['elapsed_sec'].mean()/60:.2f} phút"
        )

    # Vẽ overlay đường biên Pareto Baseline vs Cải tiến
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(b_par["latency_median_ms"], b_par["map"] * 100.0, marker="s", linestyle="--", linewidth=2, color="#1f77b4", label="Baseline Uniform Pareto")
    ax.plot(p_par["latency_median_ms"], p_par["map"] * 100.0, marker="o", linewidth=2.5, color="#2ca02c", label=f"{PAB_EXP_TAG} Pareto")
    ax.set_title(f"Pareto Frontier A/B Comparison: Uniform vs {PAB_EXP_TAG} (VOC2007 Valid)", fontweight="bold")
    ax.set_xlabel("PyTorch Eager Latency (ms, median with 200ms buffer)")
    ax.set_ylabel("mAP@50:95 (%)")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="lower right")
    ab_png = OUTPUT_DIR / "grid_search" / f"ab_pareto_uniform_vs_{PAB_EXP_TAG}.png"
    fig.tight_layout()
    fig.savefig(ab_png, dpi=300)
    plt.close(fig)
    display(Image(filename=str(ab_png)))
else:
    print("Cần có đủ cả kết quả Grid Search của Baseline và bản cải tiến để vẽ bảng đối chứng A/B.")


## 8. Đồng bộ tổng hợp về đúng tên file trên `RFDETR_Checkpoints/results_cocoinit/`


In [ ]:
import shutil

sync_pairs = [
    (OUTPUT_DIR / "grid_search" / "nas_validation_results.csv", DRIVE_RESULTS / "nas_validation_results_cocoinit.csv"),
    (OUTPUT_DIR / "grid_search" / "pareto_frontier.csv", DRIVE_RESULTS / "pareto_frontier_cocoinit.csv"),
    (OUTPUT_DIR / "grid_search" / "nas_pareto_frontier.png", DRIVE_RESULTS / "nas_pareto_frontier.png"),
    (OUTPUT_DIR / "deployment" / "throttling_comparison.png", DRIVE_RESULTS / "throttling_comparison.png"),
    (OUTPUT_DIR / "baselines" / "fair_baseline_comparison_test.csv", DRIVE_RESULTS / "fair_baseline_comparison_test.csv"),
]

for src, dst in sync_pairs:
    if src.exists() and not src.is_symlink() and src.resolve() != dst.resolve():
        shutil.copy2(src, dst)
        print(f"[Synced] {src.name} -> {dst}")

print("\n✅ Danh sách file hiện có trong results_cocoinit/:")
for p in sorted(DRIVE_RESULTS.rglob("*")):
    if p.is_file():
        print("  -", p.relative_to(DRIVE_RESULTS))
